In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [ ]:
# Define augmentation pipeline according to SimCLR paper

transform = transforms.Compose([
    transforms.RandomResizedCrop(size=32, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

In [ ]:
# Define a wrapper class essentially so augmentation is applied twice -> results in two independent views per image.

class TwoViewDataset:
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform
    
    def __len__(self):
        return len(self.base_dataset)
    
    def __getitem__(self, index):
        img, label = self.base_dataset[index]
        view1 = self.transform(img)
        view2 = self.transform(img)
        return (view1, view2), label

In [ ]:
# Load base CIFAR10
base_cifar10 = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=None)

# Wrap it -> create trainset and its DataLoader!
trainset = TwoViewDataset(base_dataset=base_cifar10, transform=transform)
trainloader = DataLoader(trainset, batch_size=256, shuffle=True, num_workers=2)

In [ ]:
import torch.nn as nn

In [4]:
# First load the ResNet-18 classifier
resnet18 = torchvision.models.resnet18(weights=None, progress=True)

# Reduce kernel size and stride as CIFAR10 images are very small
resnet18.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False) 

# Remove max pooling layer for same reason
resnet18.maxpool = nn.Identity() 

# Remove final classification layer (as embeddings are in penultimate layer)
resnet18.fc = nn.Identity()

In [ ]:
# Define projection head (only used during training)
class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(512, 512)
        self.bn = nn.BatchNorm1d(512)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(512, 128)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [ ]:
# Define SimCLR model (a wrapper of the ResNet-18 and projection head)

class SimCLR(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = resnet18
        self.projection = ProjectionHead()

    def forward(self, x1, x2):
        h1 = self.encoder(x1)  # (batch_size, 512)
        h2 = self.encoder(x2)  # (batch_size, 512)

        z1 = self.projection(h1)  # (batch_size, 128)
        z2 = self.projection(h2)  # (batch_size, 128)

        return z1, z2

    # Use after training for embedding extraction
    def get_embedding(self, x):
        return self.encoder(x)
        

In [ ]:
# TODO: define nt-xent loss function
# TODO: what optimiser? just SGD?